# LahjaLens — expériences d'amélioration du sentiment

Ce notebook compare des représentations mieux adaptées à la Darija : TF-IDF de mots, TF-IDF de caractères et combinaison mots + caractères. Il utilise les splits AfriSenti existants et le cluster Spark.

In [ ]:
from datetime import datetime, timezone
import json
import re
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, RegexTokenizer, VectorAssembler
from pyspark.sql import SparkSession, functions as F, types as T

INPUT_DIR = '/opt/lahjalens/data/processed/afrisenti/v1'
HDFS_BASE = 'hdfs://namenode:9000/data/lahjalens/models/afrisenti_v1'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
spark = SparkSession.builder.appName('LahjaLens-Sentiment-Experiments').master('spark://spark-master:7077').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

## 1. Audit des lignes réellement lues par Spark

Cette étape rend visible toute différence entre l'audit Python et le lecteur CSV Spark.

In [ ]:
def read_split(split):
    return (spark.read.options(header=True, mode='FAILFAST', quote=chr(34), escape=chr(34), multiLine=True).csv(f'{INPUT_DIR}/{split}.csv')
            .select(F.trim('comment_text').alias('comment_text'), F.col('label').cast('double').alias('label'), 'record_id')
            .where(F.col('comment_text').isNotNull() & (F.length('comment_text') > 0))
            .where(F.col('label').isin([0.0, 1.0, 2.0])))

train = read_split('train').cache()
dev = read_split('dev').cache()
test = read_split('test').cache()
loaded_counts = {name: frame.count() for name, frame in [('train', train), ('dev', dev), ('test', test)]}
print(loaded_counts)
train.groupBy('label').count().orderBy('label').show()

## 2. Construire les variables et les poids de classe

Les caractères capturent des variantes d'écriture arabe/Arabizi. La négation et les emojis restent dans le texte.

In [ ]:
@F.udf(returnType=T.ArrayType(T.StringType()))
def char_ngrams(text):
    text = re.sub(r'\s+', '_', text or '')
    return [text[index:index + 3] for index in range(max(0, len(text) - 2))]

def add_features(frame):
    return frame.withColumn('char_tokens', char_ngrams(F.col('comment_text')))

train = add_features(train)
dev = add_features(dev)
test = add_features(test)
counts = {int(row['label']): row['count'] for row in train.groupBy('label').count().collect()}
total = sum(counts.values())
weights = {label: total / (3.0 * count) for label, count in counts.items()}
mapping = F.create_map(*[item for label, weight in weights.items() for item in (F.lit(float(label)), F.lit(float(weight)))])
train = train.withColumn('class_weight', mapping[F.col('label')])
print(weights)

## 3. Définir les expériences

Nous comparons deux Logistic Regression pondérées et un Naive Bayes.

In [ ]:
word_tokenizer = RegexTokenizer(inputCol='comment_text', outputCol='word_tokens', pattern='\\s+', gaps=True, minTokenLength=1)
word_tf = HashingTF(inputCol='word_tokens', outputCol='word_raw', numFeatures=1 << 17)
word_idf = IDF(inputCol='word_raw', outputCol='word_features', minDocFreq=2)
char_tf = HashingTF(inputCol='char_tokens', outputCol='char_raw', numFeatures=1 << 17)
char_idf = IDF(inputCol='char_raw', outputCol='char_features', minDocFreq=2)
assembler = VectorAssembler(inputCols=['word_features', 'char_features'], outputCol='features')
lr = LogisticRegression(labelCol='label', featuresCol='features', weightCol='class_weight', maxIter=30, regParam=0.1, family='multinomial')
lr_word = LogisticRegression(labelCol='label', featuresCol='word_features', weightCol='class_weight', maxIter=30, regParam=0.1, family='multinomial')
nb = NaiveBayes(labelCol='label', featuresCol='word_features', smoothing=1.0)
word_lr_pipeline = Pipeline(stages=[word_tokenizer, word_tf, word_idf, lr_word])
word_char_lr_pipeline = Pipeline(stages=[word_tokenizer, word_tf, word_idf, char_tf, char_idf, assembler, lr])
word_nb_pipeline = Pipeline(stages=[word_tokenizer, word_tf, word_idf, nb])

## 4. Évaluer avec le macro-F1

Le macro-F1 donne le même poids aux trois classes, contrairement au F1 pondéré.

In [ ]:
def evaluate(predictions, split):
    predictions = predictions.cache()
    rows = predictions.count()
    accuracy = float(MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy').evaluate(predictions))
    weighted_f1 = float(MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1').evaluate(predictions))
    matrix = {(int(row['label']), int(row['prediction'])): row['count'] for row in predictions.groupBy('label', 'prediction').count().collect()}
    f1_values = {}
    for label, name in ((0, 'negative'), (1, 'neutral'), (2, 'positive')):
        tp = matrix.get((label, label), 0)
        actual = sum(count for (truth, _), count in matrix.items() if truth == label)
        predicted = sum(count for (_, guess), count in matrix.items() if guess == label)
        precision = tp / predicted if predicted else 0.0
        recall = tp / actual if actual else 0.0
        f1_values[name] = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'split': split, 'rows': rows, 'accuracy': accuracy, 'weighted_f1': weighted_f1, 'macro_f1': sum(f1_values.values()) / 3.0, 'per_class_f1': f1_values}

experiments = [('lr_word', word_lr_pipeline), ('lr_word_char', word_char_lr_pipeline), ('naive_bayes_word', word_nb_pipeline)]
results = []
models = {}
for name, pipeline in experiments:
    print('TRAINING', name)
    fitted = pipeline.fit(train)
    models[name] = fitted
    dev_metrics = evaluate(fitted.transform(dev), 'dev')
    test_metrics = evaluate(fitted.transform(test), 'test')
    results.append({'model': name, 'dev': dev_metrics, 'test': test_metrics})
    print(json.dumps(results[-1], ensure_ascii=False, indent=2))

## 5. Sauvegarder le meilleur modèle et les métriques

La sélection se fait sur `dev`, jamais sur `test`.

In [ ]:
best = max(results, key=lambda item: item['dev']['macro_f1'])
best_path = f'{HDFS_BASE}/experiments/{RUN_ID}/{best["model"]}'
models[best['model']].write().overwrite().save(best_path)
report = {'run_id': RUN_ID, 'dataset': 'afrisenti', 'loaded_counts': loaded_counts, 'class_weights': weights, 'results': results, 'selected_model': best['model'], 'selected_model_path': best_path}
spark.sparkContext.parallelize([json.dumps(report, ensure_ascii=False)], 1).saveAsTextFile(f'{HDFS_BASE}/experiments/{RUN_ID}/metrics')
print(json.dumps(report, ensure_ascii=False, indent=2))
spark.stop()